# Entrenamiento de Modelos de Machine Learning

1. **Preprocesar los datos de forma científica** usando Pipelines y ColumnTransformer de Scikit-Learn (para evitar la fuga de datos del conjunto de prueba).
2. **Entrenar y ajustar 5 algoritmos** (Regresión Logística, Árbol de Decisión, Random Forest, XGBoost y una Red Neuronal con Keras/TensorFlow).
3. **Serializar y guardar** los modelos para producción.


In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

# Añadir directorio raíz al path para usar src
sys.path.append(str(Path.cwd().parent))

## 1. Cargar y Preprocesar los Datos

Primero dividimos los datos en **Entrenamiento (Train)** y **Prueba (Test)**.

El conjunto de prueba representa el futuro del modelo. Nunca debemos preprocesar todo el dataset a la vez: si calculamos la mediana de una columna sobre el dataset completo para rellenar nulos, el valor del test influiría en el de train, causando una fuga de información sutil.

> **Solución**: El preprocesador (ColumnTransformer) vive dentro del Pipeline. Al llamar a pipeline.fit(X_train, y_train), el Pipeline ajusta el preprocesador usando únicamente los datos de entrenamiento y lo aplica automáticamente al conjunto de prueba en la evaluación, sin contaminarlo.


In [2]:
from src.data_loader import load_data, preprocess_and_split

# Cargar y preprocesar/dividir datos usando el pipeline robusto de src
df = load_data()
X_train, X_test, y_train, y_test, preprocessor = preprocess_and_split(df)

print(f"Conjunto de Entrenamiento: {X_train.shape[0]:,} filas")
print(f"Conjunto de Prueba: {X_test.shape[0]:,} filas")

Conjunto de Entrenamiento: 95,512 filas
Conjunto de Prueba: 23,878 filas


### Construcción del Preprocesador

Usamos las listas de columnas explícitas definidas en src/config.py en lugar de inferir los tipos por columna. Esto garantiza que el preprocesador sea reproducible y coherente con el pipeline de producción: las mismas columnas, en el mismo orden, siempre.

* Para numéricas: rellenar vacíos con la mediana y normalizar la escala (StandardScaler).
* Para categóricas: rellenar vacíos con "Unknown" y aplicar codificación binaria (OneHotEncoder).
* arrival_date_year se descarta mediante remainder="drop". El dataset cubre años parciales (2015 solo jul-dic, 2017 solo ene-ago), lo que genera una correlacion espuria con la variable objetivo que no se repetiría en producción.


In [3]:
# El preprocesador es un Pipeline completo que incluye:
# - Ingeniería de Características (FeatureEngineer)
# - ColumnTransformer con normalización numérica y OneHotEncoder categórico
# - Conversión de tipos mixtos (CastToString)
preprocessor

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('features', ...), ('col_transform', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

## 2. Entrenamiento de los 5 Modelos

Entrenaremos los 5 modelos encapsulando cada uno en un Pipeline que une el preprocesador con el clasificador. La red neuronal se incluye en el mismo bucle que el resto gracias a un adaptador que implementa la interfaz de Scikit-Learn (KerasMLPClassifier, definido en src/model_trainer.py).


In [4]:
from src.model_trainer import train_all_models

# Entrenar todos los modelos usando el pipeline orquestado en model_trainer
trained_pipelines = train_all_models(preprocessor, X_train, y_train, tune_xgboost=False)

KeyboardInterrupt: 

### Nota sobre la serialización

Todos los modelos, incluida la red neuronal, se guardan como un único archivo .pkl que contiene tanto el preprocesador como el clasificador. Esto simplifica la carga en produccion: un solo joblib.load es suficiente para obtener un objeto listo para recibir datos en crudo y devolver una prediccion.


## 3. Bonus — Ajuste de Hiperparámetros de XGBoost con RandomizedSearchCV\n
\n
Los hiperparámetros fijados en el paso anterior son razonables, pero no necesariamente óptimos. **RandomizedSearchCV** explora aleatoriamente un espacio de combinaciones y selecciona la que maximiza el ROC-AUC en validación cruzada (3 folds), evitando el coste de un GridSearchCV exhaustivo.\n

In [5]:
from src.model_trainer import train_all_models

# Entrenar y optimizar XGBoost con RandomizedSearchCV (3 folds, 30 combinaciones)\n
# Esto sobrescribirá models/xgboost.pkl con la mejor combinación encontrada.\n
trained_pipelines_opt = train_all_models(preprocessor, X_train, y_train, tune_xgboost=True)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 